# 🐍 Modul 06 — Analisis Saham dengan Python

> Dari membaca CSV sampai menghitung return, risiko, dan membandingkan dengan IHSG.

**Estimasi:** 32 menit · **Pelajaran:** 2 ·
**Soal kuis:** 6

[← Kembali ke panduan belajar](00_Panduan_Belajar.ipynb)

## Cara memakai notebook ini

1. Baca satu pelajaran sampai selesai.
2. Jawab tiga soal tanpa membuka kunci jawaban.
3. Buka kunci, lalu jelaskan ulang bagian yang salah dengan kata sendiri.
4. Lanjut ke pelajaran berikutnya jika sudah memahami intinya.

Materi ini untuk dibaca. Untuk latihan Python yang dijalankan dengan data
proyek, buka folder `notebooks/`.

## Daftar pelajaran

1. **Membaca dan membersihkan data dengan Pandas** — 15 menit
2. **Return, volatilitas, drawdown, dan benchmark** — 17 menit

---

## Pelajaran 1: Membaca dan membersihkan data dengan Pandas

**Estimasi:** 15 menit

**Tujuan pembelajaran:** Memeriksa bentuk, tipe, missing, duplikat, dan tanggal sebelum menghitung apa pun.

### Langkah pertama: kenali data

Sebelum menghitung return, pastikan kamu tahu isi datanya:

```python
import pandas as pd

df = pd.read_csv("data/processed/historical_prices.csv")
print(df.shape)
print(df.head())
print(df.dtypes)
print(df.isna().sum())
print(df.duplicated(subset=["date", "ticker"]).sum())
```

`shape` memberi jumlah baris dan kolom. `head()` melihat contoh isi. `dtypes` menunjukkan tipe data. `isna().sum()` menghitung nilai hilang. Duplikat dicek berdasarkan kombinasi tanggal dan ticker, bukan seluruh baris.

### Perbaiki tipe data

Tanggal harus bertipe datetime, bukan teks. Harga dan volume harus angka. Jika tidak, operasi seperti `pct_change()` atau `groupby()` bisa gagal atau menghasilkan urutan yang salah.

```python
df["date"] = pd.to_datetime(df["date"], errors="coerce")
kolom_angka = ["open", "high", "low", "close", "adj_close", "volume"]
for kolom in kolom_angka:
    df[kolom] = pd.to_numeric(df[kolom], errors="coerce")
df = df.dropna(subset=["date", "ticker", "adj_close"])
df = df.drop_duplicates(subset=["date", "ticker"], keep="last")
df = df.sort_values(["ticker", "date"])
```

Gunakan `errors="coerce"` agar nilai aneh menjadi NaN dan bisa diperiksa, bukan langsung membuat program berhenti.

### Validasi logika harga

Periksa apakah harga masuk akal: harga penutupan positif, `high` tidak lebih rendah dari `low`, dan volume tidak negatif. Jangan otomatis menghapus semua baris yang mencurigakan; tandai dulu untuk diperiksa.

Pada data historis proyek ini, validasi dilakukan di `src/data_cleaning.py` dan ringkasannya ditampilkan di halaman Data Overview.

### Jangan mengisi data secara asal

`ffill()` dapat membuat grafik terlihat mulus, tetapi juga menyembunyikan kenyataan bahwa saham tidak diperdagangkan. Untuk return, gunakan harga yang benar-benar ada dan biarkan NaN. Untuk analisis korelasi, gunakan pasangan tanggal yang tersedia.

Jika kamu mengisi harga berhari-hari ke depan, kamu bisa menciptakan return nol palsu dan menurunkan volatilitas secara tidak wajar.

### Analogi

> Membersihkan data seperti mencuci sayur sebelum memasak. Kamu membuang kotoran, tetapi tidak memotong semua bagian yang bentuknya aneh tanpa memeriksa apakah itu bagian yang penting. Data krisis yang ekstrem justru bisa menjadi pelajaran berharga.

### Contoh

File memiliki 1.000 baris, 3 tanggal duplikat, dan 40 harga `adj_close` kosong. Jika langsung dihitung, return bisa error atau menghasilkan angka aneh.

Setelah tanggal diubah ke datetime, duplikat dihapus, dan baris tanpa harga dibuang, tersisa 995 baris. Perubahannya dicatat di laporan kualitas data, bukan disembunyikan.

### Penerapan

Notebook `01_data_understanding.ipynb` dan `02_data_cleaning.ipynb` menjalankan langkah ini pada data proyek. Bandingkan output notebook dengan halaman Data Overview.

### Contoh kode Python

Contoh berikut untuk dibaca dan dicoba. Untuk praktik yang benar-benar
dijalankan dengan data proyek, gunakan notebook di folder `notebooks/`.

```python
df = df.sort_values(["ticker", "date"])
df["return"] = df.groupby("ticker")["adj_close"].pct_change()
print(df.groupby("ticker")["return"].describe())
```

### Kesalahan yang perlu dihindari

- Membaca tanggal sebagai teks lalu mengurutkannya secara alfabet.
- Menghapus semua baris dengan nilai hilang tanpa memeriksa polanya.
- Mengisi harga saham berhari-hari dengan forward-fill.
- Menghitung return dari harga mentah yang belum disesuaikan.

### Ringkasan

- Kenali bentuk, tipe, missing, dan duplikat sebelum analisis.
- Ubah tanggal dan angka ke tipe yang benar.
- Validasi harga dan volume secara logis.
- Jangan mengisi atau menghapus data secara sembarangan.
- Simpan data mentah dan hasil olahan secara terpisah.

### Kuis — 3 soal

Jawab dulu tanpa membuka kunci di bawah. Nilai **minimal 2 dari 3 benar**
untuk dianggap lulus pelajaran ini.

**1. Apa fungsi `df.shape`?**

- A. Menampilkan jumlah baris dan kolom
- B. Menghapus duplikat
- C. Mengubah tanggal menjadi angka
- D. Menghitung return

**2. Mengapa tanggal perlu diubah dengan `pd.to_datetime`?**

- A. Agar bisa diurutkan dan diolah sebagai waktu dengan benar
- B. Agar harga saham naik
- C. Agar file lebih kecil
- D. Agar tidak ada missing value

**3. Apa risiko mengisi harga yang hilang dengan forward-fill tanpa batas?**

- A. Membuat data terlihat lebih lengkap daripada kenyataan dan return menjadi palsu
- B. Membuat harga selalu turun
- C. Menghapus semua dividen
- D. Mengubah lot menjadi 1.000

<details>
<summary>Kunci jawaban dan pembahasan</summary>

- **1. A** — `shape` mengembalikan tuple (jumlah baris, jumlah kolom).
- **2. A** — Tipe datetime memungkinkan pengurutan, pemotongan periode, dan perhitungan selisih waktu yang benar.
- **3. A** — Forward-fill panjang menciptakan harga datar palsu, return nol, dan volatilitas yang terlalu rendah.

</details>

---

## Pelajaran 2: Return, volatilitas, drawdown, dan benchmark

**Estimasi:** 17 menit

**Tujuan pembelajaran:** Menghitung metrik inti dan menafsirkannya dengan jujur.

### Return harian dan cumulative return

**Return harian** = (harga hari ini ÷ harga kemarin) − 1.

**Cumulative return** = (harga akhir ÷ harga awal) − 1. Ini menunjukkan total perubahan selama periode, bukan rata-rata per tahun.

Contoh: Rp1.000 menjadi Rp1.200 dalam 2 tahun. Cumulative return = 20%. Tetapi CAGR (pertumbuhan tahunan majemuk) = (1.200 ÷ 1.000)^(1/2) − 1 = 9,54% per tahun, bukan 10% karena efek majemuk.

### CAGR dan moving average

**CAGR** = (harga akhir ÷ harga awal)^(1 ÷ jumlah tahun) − 1. CAGR berguna untuk membandingkan periode berbeda, tetapi tetap berdasarkan titik awal dan akhir; perjalanan di tengah tidak terlihat.

**Moving average (MA)** adalah rata-rata harga selama beberapa hari terakhir. MA 20 memakai 20 hari; MA 200 memakai 200 hari. MA membantu melihat arah tren, tetapi selalu tertinggal dari harga karena memakai data masa lalu. Jangan menganggap MA sebagai ramalan pasti.

### Volatilitas dan drawdown

**Volatilitas tahunan** = simpangan baku return harian × √252. Angka ini menggambarkan seberapa besar naik-turun return, bukan batas kerugian.

**Drawdown** = harga ÷ puncak tertinggi sebelumnya − 1. **Maximum drawdown** adalah nilai terendah drawdown. Jika saham turun 40% dari puncak, dibutuhkan kenaikan 66,7% untuk kembali.

Kedua metrik ini dihitung di `src/risk_analysis.py` dan ditampilkan di halaman Analisis Risiko.

### Membandingkan dengan IHSG

Benchmark membuat penilaian lebih adil. Jika sahammu naik 10% tetapi IHSG naik 20% pada periode yang sama, sahammu tertinggal dari pasar.

Syarat perbandingan: periode sama, harga adjusted, dan mata uang sama. Untuk beberapa saham, gunakan tanggal awal yang sama. Jangan membandingkan return 3 tahun saham dengan IHSG 1 tahun.

**Return relatif** = return saham − return benchmark. Ini bukan jaminan masa depan, hanya konteks historis.

### Analogi

> Return seperti jarak yang ditempuh, CAGR seperti kecepatan rata-rata tahunan, volatilitas seperti seberapa sering arah berubah, dan drawdown seperti seberapa dalam kamu pernah jatuh ke lubang. Semuanya diperlukan untuk menggambarkan perjalanan, bukan hanya garis finis.

### Contoh

Harga awal Rp1.000, harga akhir Rp1.500 setelah 3 tahun.

- Cumulative return = 50%
- CAGR = (1.500 ÷ 1.000)^(1/3) − 1 = 14,47% per tahun
- Jika selama periode pernah turun ke Rp700 dari puncak Rp1.600, drawdown = (700 ÷ 1.600) − 1 = −56,25%
- Volatilitas tahunan 30% berarti naik-turun harian cukup besar, bukan berarti pasti rugi 30%.

Saham dengan return 50% tetapi drawdown 56% terasa jauh lebih menegangkan daripada angka return-nya.

### Penerapan

Di halaman **Performa Saham**, pilih beberapa saham dan IHSG, lalu bandingkan grafik ternormalisasi dan tabel metrik. Di halaman **Risiko**, lihat drawdown dan distribusi return.

### Contoh kode Python

Contoh berikut untuk dibaca dan dicoba. Untuk praktik yang benar-benar
dijalankan dengan data proyek, gunakan notebook di folder `notebooks/`.

```python
import numpy as np

harga = df.set_index("date")["adj_close"]
ret = harga.pct_change().dropna()
cumulative = harga.iloc[-1] / harga.iloc[0] - 1
vol = ret.std() * np.sqrt(252)
drawdown = (harga / harga.cummax() - 1).min()
print(cumulative, vol, drawdown)
```

### Kesalahan yang perlu dihindari

- Membandingkan return periode berbeda.
- Menganggap CAGR sama dengan rata-rata return tahunan sederhana.
- Menganggap volatilitas tinggi berarti pasti bangkrut.
- Menggunakan moving average sebagai jaminan beli/jual.
- Tidak memasukkan biaya transaksi dalam evaluasi strategi.

### Ringkasan

- Return harian mengukur perubahan satu hari; cumulative return mengukur total periode.
- CAGR mempertimbangkan efek majemuk.
- Volatilitas mengukur naik-turun; drawdown mengukur penurunan dari puncak.
- Bandingkan dengan benchmark pada periode yang sama.
- Metrik historis tidak menjamin hasil masa depan.

### Kuis — 3 soal

Jawab dulu tanpa membuka kunci di bawah. Nilai **minimal 2 dari 3 benar**
untuk dianggap lulus pelajaran ini.

**1. Harga naik dari Rp1.000 ke Rp1.210 dalam 2 tahun. Berapa CAGR?**

- A. 10%
- B. 21%
- C. 9,54%
- D. 5%

**2. Apa arti maximum drawdown −40%?**

- A. Harga pernah turun 40% dari puncak sebelumnya
- B. Harga pasti turun 40% tahun depan
- C. Volatilitas harian 40%
- D. Dividen turun 40%

**3. Mengapa perbandingan dengan IHSG harus memakai periode yang sama?**

- A. Agar kesimpulan performa relatif adil
- B. Agar harga saham otomatis naik
- C. Karena IHSG tidak punya tanggal
- D. Karena periode tidak memengaruhi return

<details>
<summary>Kunci jawaban dan pembahasan</summary>

- **1. A** — (1.210/1.000)^(1/2) − 1 = 0,10 atau 10% per tahun. Cumulative return 21% dibagi dua tahun bukan CAGR.
- **2. A** — Drawdown dihitung dari puncak ke lembah. −40% berarti penurunan terburuk sebesar 40% dari titik puncak pada periode itu.
- **3. A** — Periode berbeda membuat perbandingan tidak setara; kondisi pasar bisa sangat berbeda.

</details>

---

## Checklist modul

- [ ] Saya membaca kedua pelajaran sampai selesai.
- [ ] Saya menjawab enam soal tanpa membuka kunci lebih dulu.
- [ ] Saya bisa menjelaskan inti modul dengan kata sendiri.
- [ ] Saya mencatat bagian yang masih belum jelas.

**Modul berikutnya:** [📝 Modul 07 — Simulasi dan Paper Trading](Modul_07_Simulasi_dan_Paper_Trading.ipynb)

> **Pengingat:** materi ini bukan rekomendasi investasi dan tidak menjanjikan
> keuntungan. Jangan memakai utang, margin, atau uang kebutuhan untuk berinvestasi.